# Platform Style Analysis: Blog vs. LinkedIn

Visualize structural differences extracted by `research/analyze.py`.
Use findings to validate and tune the LinkedIn rubric in `eval/voice_rubric.py`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load analysis results
results_path = Path('../research/analysis_results.json')
with open(results_path) as f:
    results = json.load(f)

print(f"Loaded results: {len(results['blog']['n_samples'])} blog, {len(results['linkedin']['n_samples'])} linkedin samples")

In [ ]:
# Compare hook velocity (words until first sentence ends)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

blog_per_sample = [s.get('hook_velocity_words') for s in results['per_sample']['blog'] if s.get('hook_velocity_words')]
linkedin_per_sample = [s.get('hook_velocity_words') for s in results['per_sample']['linkedin'] if s.get('hook_velocity_words')]

axes[0].hist(blog_per_sample, bins=15, alpha=0.6, label='Blog', edgecolor='black')
axes[0].hist(linkedin_per_sample, bins=15, alpha=0.6, label='LinkedIn', edgecolor='black')
axes[0].set_xlabel('Words in First Sentence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Hook Velocity (words until first sentence ends)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Summary stats
blog_hook = results['blog'].get('hook_velocity_words', {})
linkedin_hook = results['linkedin'].get('hook_velocity_words', {})
axes[1].barh(['Blog Mean', 'LinkedIn Mean'], 
             [blog_hook.get('mean', 0), linkedin_hook.get('mean', 0)],
             color=['steelblue', 'darkorange'])
axes[1].set_xlabel('Words')
axes[1].set_title('Hook Velocity: Mean Comparison')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Blog hook velocity: mean={blog_hook.get('mean')}, median={blog_hook.get('median')}")
print(f"LinkedIn hook velocity: mean={linkedin_hook.get('mean')}, median={linkedin_hook.get('median')}")

In [ ]:
# Compare paragraph structure
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

blog_spp = [s.get('sentences_per_paragraph_mean') for s in results['per_sample']['blog'] if s.get('sentences_per_paragraph_mean')]
linkedin_spp = [s.get('sentences_per_paragraph_mean') for s in results['per_sample']['linkedin'] if s.get('sentences_per_paragraph_mean')]

axes[0].hist(blog_spp, bins=12, alpha=0.6, label='Blog', edgecolor='black')
axes[0].hist(linkedin_spp, bins=12, alpha=0.6, label='LinkedIn', edgecolor='black')
axes[0].set_xlabel('Sentences per Paragraph (mean)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Paragraph Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

blog_spp_stats = results['blog'].get('sentences_per_paragraph_mean', {})
linkedin_spp_stats = results['linkedin'].get('sentences_per_paragraph_mean', {})
axes[1].barh(['Blog Mean', 'LinkedIn Mean'],
             [blog_spp_stats.get('mean', 0), linkedin_spp_stats.get('mean', 0)],
             color=['steelblue', 'darkorange'])
axes[1].set_xlabel('Sentences')
axes[1].set_title('Sentences per Paragraph: Mean Comparison')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Blog sentences/para: mean={blog_spp_stats.get('mean')}, median={blog_spp_stats.get('median')}")
print(f"LinkedIn sentences/para: mean={linkedin_spp_stats.get('mean')}, median={linkedin_spp_stats.get('median')}")

In [ ]:
# Compare line break density
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

blog_lbd = [s.get('line_break_density') for s in results['per_sample']['blog'] if s.get('line_break_density') is not None]
linkedin_lbd = [s.get('line_break_density') for s in results['per_sample']['linkedin'] if s.get('line_break_density') is not None]

axes[0].hist(blog_lbd, bins=15, alpha=0.6, label='Blog', edgecolor='black')
axes[0].hist(linkedin_lbd, bins=15, alpha=0.6, label='LinkedIn', edgecolor='black')
axes[0].set_xlabel('Blank lines per 100 words (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Line Break Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

blog_lbd_stats = results['blog'].get('line_break_density', {})
linkedin_lbd_stats = results['linkedin'].get('line_break_density', {})
axes[1].barh(['Blog Mean', 'LinkedIn Mean'],
             [blog_lbd_stats.get('mean', 0), linkedin_lbd_stats.get('mean', 0)],
             color=['steelblue', 'darkorange'])
axes[1].set_xlabel('% per 100 words')
axes[1].set_title('Line Break Density: Mean Comparison')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Blog line break density: mean={blog_lbd_stats.get('mean')}, median={blog_lbd_stats.get('median')}")
print(f"LinkedIn line break density: mean={linkedin_lbd_stats.get('mean')}, median={linkedin_lbd_stats.get('median')}")

In [ ]:
# Feature presence comparison (boolean features)
features_to_compare = [
    'has_direct_address',
    'ends_with_question',
    'ends_with_cta',
    'first_sentence_has_reaction',
]

blog_stats = results['blog']
linkedin_stats = results['linkedin']

fig, ax = plt.subplots(figsize=(10, 6))

features = []
blog_pcts = []
linkedin_pcts = []

for feature in features_to_compare:
    if feature in blog_stats and feature in linkedin_stats:
        features.append(feature.replace('_', ' ').title())
        blog_pcts.append(blog_stats[feature].get('present_pct', 0))
        linkedin_pcts.append(linkedin_stats[feature].get('present_pct', 0))

x = range(len(features))
width = 0.35

ax.barh([i - width/2 for i in x], blog_pcts, width, label='Blog', color='steelblue')
ax.barh([i + width/2 for i in x], linkedin_pcts, width, label='LinkedIn', color='darkorange')

ax.set_yticks(x)
ax.set_yticklabels(features)
ax.set_xlabel('% of Posts with Feature')
ax.set_title('Feature Presence: Blog vs LinkedIn')
ax.legend()
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print("\n=== PLATFORM STYLE COMPARISON ===")
print(f"\nBlog samples: {results['blog']['n_samples']}")
print(f"LinkedIn samples: {results['linkedin']['n_samples']}")

comparison_data = {
    'Metric': [
        'Hook velocity (words)',
        'Sentences per para',
        'Line break density (%)',
        'Has "you"',
        'Ends with question',
        'Ends with CTA',
        'First sent has reaction',
    ],
    'Blog Mean': [
        f"{results['blog'].get('hook_velocity_words', {}).get('mean', 'N/A')}",
        f"{results['blog'].get('sentences_per_paragraph_mean', {}).get('mean', 'N/A')}",
        f"{results['blog'].get('line_break_density', {}).get('mean', 'N/A')}",
        f"{results['blog'].get('has_direct_address', {}).get('present_pct', 'N/A')}%",
        f"{results['blog'].get('ends_with_question', {}).get('present_pct', 'N/A')}%",
        f"{results['blog'].get('ends_with_cta', {}).get('present_pct', 'N/A')}%",
        f"{results['blog'].get('first_sentence_has_reaction', {}).get('present_pct', 'N/A')}%",
    ],
    'LinkedIn Mean': [
        f"{results['linkedin'].get('hook_velocity_words', {}).get('mean', 'N/A')}",
        f"{results['linkedin'].get('sentences_per_paragraph_mean', {}).get('mean', 'N/A')}",
        f"{results['linkedin'].get('line_break_density', {}).get('mean', 'N/A')}",
        f"{results['linkedin'].get('has_direct_address', {}).get('present_pct', 'N/A')}%",
        f"{results['linkedin'].get('ends_with_question', {}).get('present_pct', 'N/A')}%",
        f"{results['linkedin'].get('ends_with_cta', {}).get('present_pct', 'N/A')}%",
        f"{results['linkedin'].get('first_sentence_has_reaction', {}).get('present_pct', 'N/A')}%",
    ]
}

df = pd.DataFrame(comparison_data)
print("\n" + df.to_string(index=False))

print("\n=== INTERPRETATION ===")
print("Use the above metrics to validate the LinkedIn rubric in eval/voice_rubric.py:")
print("- If LinkedIn posts have significantly shorter hooks → hook check is valid")
print("- If LinkedIn posts have denser line breaks → line break check is valid")
print("- If feature presence diverges by >20% → consider adjusting rubric weights")